# The Used Car Bargain Hunter

The final step is to launch the User Interface (UI) using Gradio.

This notebook sets up the final dashboard for the deal hunting agent.

In [ ]:
import gradio as gr
from deal_agent_framework import DealAgentFramework
from agents.deals import Opportunity, Deal

In [ ]:
agent_framework = DealAgentFramework()
agent_framework.init_agents_as_needed()

In [ ]:
with gr.Blocks(title="Used Car Bargain Hunter", fill_width=True) as ui:
        
    # Provide a placeholder initial deal to establish the table schema
    initial_deal = Deal(product_description="Waiting for deals...", price=0.0, url="#")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=0.0, discount=0.0)
    opportunities = gr.State([initial_opportunity])
    def get_table(opps):
        return [[opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url] for opp in opps]
    def do_select(opportunities, selected_index: gr.SelectData):
        row = selected_index.index[0]
        opportunity = opportunities[row]
        agent_framework.planner.messenger.alert(opportunity)
        return gr.Info("Alert sent successfully!")
    # --- NEW LOGIC: Trigger the scanning agent ---
    def run_scan():
        gr.Info("Scanning the internet for deals... This might take a minute.")
        new_opps = agent_framework.run()
        if not new_opps:
            return [initial_opportunity]
        return new_opps
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">Used Car Bargain Hunter - Agentic Artificial Intelligence (AI)</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Autonomous agent framework that finds underpriced vehicle deals using a Retrieval-Augmented Generation (RAG) pipeline.</div>')
    
    # --- NEW BUTTON ---
    with gr.Row():
        scan_button = gr.Button("🔍 Scan for Deals Now", variant="primary")
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Listed Price", "Estimated True Value", "Discount Amount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )
    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities], outputs=[])
    
    # --- NEW LOGIC: Connect the button to the UI table ---
    scan_button.click(fn=run_scan, inputs=[], outputs=[opportunities]).then(
        fn=get_table, inputs=[opportunities], outputs=[opportunities_dataframe]
    )

In [ ]:
ui.launch(inbrowser=True)